In [2]:
import pandas as pd

In [3]:
loc = pd.read_csv('LOC.csv')
dependents = pd.read_csv('dependent_count.csv')
downloads = pd.read_csv('download_count.csv')
package_data = pd.read_csv('package_data_first_release_dates.csv')

Normalize Git tags into standardized version strings with the following function, clean_version.

In [4]:
import re

def clean_version(tag_name):
    """Remove package name prefixes before version numbers."""
    tag_name = str(tag_name)
    
    # Strategy 1: Find @ followed by a proper version pattern (digit.digit...)
    # Handles: @org/package@1.0.0 -> 1.0.0
    matches = list(re.finditer(r'@(\d+\.\d+[^\s]*)$', tag_name))
    if matches:
        return matches[-1].group(1)
    
    # Strategy 2: Find / followed by version pattern (for package/version style)
    # Handles: package-name/v1.0.0 -> 1.0.0
    match = re.search(r'/[v]?(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 3: Find _ followed by v and version pattern
    # Handles: gmv3_v2.0.0 -> 2.0.0
    match = re.search(r'_v(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 4: Handle package-version style with hyphen
    # Handles: package-1.0.0 -> 1.0.0
    match = re.search(r'-(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 5: Fallback - remove any non-digit prefix (for v1.0.0 style)
    # Handles: v1.0.0 -> 1.0.0
    match = re.search(r'^[^\d]*(\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    return tag_name

In the following code, test for semantic versioning.

In [5]:
def is_semantic_version(tag_name):
    """
    Check if tag follows semantic versioning: MAJOR.MINOR.PATCH
    Also accepts minor versions (MAJOR.MINOR) and major versions (MAJOR)
    Allows suffixes like alpha, beta, rc, post, dev, etc.
    """
    tag_name = str(tag_name)
    
    # Pattern for semantic versioning with optional suffixes
    # Matches: X.Y.Z, X.Y, or X (where X, Y, Z are numbers)
    # Allows suffixes: -alpha, .post1, a0, b1, rc2, dev3, etc.
    pattern = r'^\d+(\.\d+)?(\.\d+)?([a-zA-Z0-9\.\-]+)?$'
    
    return bool(re.match(pattern, tag_name))

In [6]:
# Apply the clean_version function
dependents['tag_name_clean'] = dependents['version'].apply(clean_version)
downloads['tag_name_clean'] = downloads['version'].apply(clean_version)
loc['tag_name_clean'] = loc['tag_name'].apply(clean_version)
package_data['tag_name_clean'] = package_data['tag_name'].apply(clean_version)

## Merge LOC data with release date package data

In [20]:
## merge package_data with loc subset based on github_repo, tag_name, and tag_commit_sha
loc_subset = loc[['github_repo', 'tag_name_clean', 'tag_commit_sha', 'code']]
loc_subset.rename(columns={'code': 'loc'}, inplace=True)

merged_data = pd.merge(package_data, 
                       loc_subset, 
                       left_on=['github_repo', 'tag_name_clean', 'tag_commit_sha'], 
                       right_on=['github_repo', 'tag_name_clean', 'tag_commit_sha'], 
                       how='left')


/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_81106/2139751832.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loc_subset.rename(columns={'code': 'loc'}, inplace=True)


## Merge Number of Dependents data with merged data from previous step

In [21]:
## now merge the dependents data with the merged_data based on package_name and tag_name_clean
merged_data = pd.merge(merged_data, 
                 dependents, 
                 left_on=['package_name', 'tag_name_clean'], 
                 right_on=['package', 'tag_name_clean'], 
                 how='left')

In [22]:
## now merge the downloads data with the merged_data based on package_name and tag_name_clean
merged_data = pd.merge(merged_data, 
                downloads,
                left_on=['package_name', 'tag_name_clean'],
                right_on=['package_name','tag_name_clean'], 
                how='left')

In [23]:
## now remove excessive columns
merged_data = merged_data.drop(columns=['version_x', 'version_y', 'package', 'status', 'ecosystem', 'release_tag_name'])

In [24]:
## calculate the version age in days
merged_data['version_age_days'] = (pd.to_datetime(merged_data['published_at']) - pd.to_datetime(merged_data['first_release_date'])).dt.days

In [26]:
merged_data.to_csv('control_data_final.csv', index=False)